
# Milestone 4 – API Data Transformation and Cleansing

## Project Continuation from Milestone 3

This notebook continues the data cleaning and transformation workflow from Milestone 3,
but instead of HTML website scraping, this milestone uses API/JSON data directly.

The notebook demonstrates:
- API extraction using requests
- JSON normalization
- Multiple transformation and cleansing steps
- Human-readable formatted dataset
- Ethical implications discussion



## Import Libraries


In [1]:

import requests
import pandas as pd
import numpy as np
from difflib import get_close_matches

pd.set_option('display.max_columns', None)



## Extract Data from API

The OpenLibrary API is used to retrieve book metadata related to data science.


In [2]:

url = "https://openlibrary.org/search.json?q=data+science"

response = requests.get(url)

print("Status Code:", response.status_code)

data = response.json()

print("Total Records Returned:", len(data['docs']))


ConnectTimeout: HTTPSConnectionPool(host='openlibrary.org', port=443): Max retries exceeded with url: /search.json?q=data+science (Caused by ConnectTimeoutError(<HTTPSConnection(host='openlibrary.org', port=443) at 0x277cdd72660>, 'Connection to openlibrary.org timed out. (connect timeout=None)'))


# Step #1 – Normalize JSON Data

The nested JSON response is converted into a pandas DataFrame to make the
dataset easier to clean, manipulate, and analyze.


In [ ]:

records = data['docs'][:100]

df = pd.json_normalize(records)

columns = [
    'title',
    'author_name',
    'first_publish_year',
    'edition_count',
    'language',
    'publisher'
]

df = df[columns]

df.head()



# Step #2 – Replace Headers

Column names are reformatted into cleaner, human-readable labels.


In [ ]:

df.columns = [
    'Title',
    'Author',
    'First Publish Year',
    'Edition Count',
    'Language',
    'Publisher'
]

df.head()



# Step #3 – Fix Missing Values

Missing values are identified and replaced with appropriate placeholder values.
Lists returned from the API are simplified into readable strings.


In [ ]:

df['Author'] = df['Author'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'Unknown'
)

df['Language'] = df['Language'].apply(
    lambda x: ', '.join(x[:3]) if isinstance(x, list) else 'Unknown'
)

df['Publisher'] = df['Publisher'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'Unknown'
)

df['First Publish Year'] = df['First Publish Year'].fillna(
    df['First Publish Year'].median()
)

df.head()



# Step #4 – Identify Outliers and Bad Data

Publication years outside realistic ranges are removed from the dataset.


In [ ]:

before_rows = len(df)

df = df[
    (df['First Publish Year'] >= 1900) &
    (df['First Publish Year'] <= 2026)
]

after_rows = len(df)

print("Rows before filtering:", before_rows)
print("Rows after filtering:", after_rows)



# Step #5 – Find and Remove Duplicates

Duplicate records based on title and author combinations are removed.


In [ ]:

before_duplicates = len(df)

df = df.drop_duplicates(subset=['Title', 'Author'])

after_duplicates = len(df)

print("Rows before duplicate removal:", before_duplicates)
print("Rows after duplicate removal:", after_duplicates)



# Step #6 – Fix Casing and Inconsistent Values

Text values are standardized using title casing for improved readability.


In [ ]:

df['Title'] = df['Title'].str.title()
df['Author'] = df['Author'].str.title()
df['Publisher'] = df['Publisher'].str.title()

df.head()



# Step #7 – Conduct Fuzzy Matching

Publisher names are standardized using fuzzy matching to reduce inconsistencies.


In [ ]:

publishers = df['Publisher'].dropna().unique().tolist()

def fuzzy_standardize(value, choices):
    match = get_close_matches(value, choices, n=1, cutoff=0.85)
    return match[0] if match else value

df['Publisher Standardized'] = df['Publisher'].apply(
    lambda x: fuzzy_standardize(x, publishers)
)

df[['Publisher', 'Publisher Standardized']].head(10)



# Final Human-Readable Dataset


In [ ]:

final_df = df.reset_index(drop=True)

print(final_df.head(20))

final_df.info()



# Ethical Implications

## What changes were made to the data?
The API JSON data was normalized into a structured DataFrame, missing values were
handled, duplicate rows were removed, casing inconsistencies were fixed, and
publisher names were standardized using fuzzy matching.

## Are there any legal or regulatory guidelines for your data or project topic?
The OpenLibrary API provides publicly accessible metadata. Users should still
follow responsible data usage and attribution practices.

## What risks could be created based on the transformations done?
Fuzzy matching could incorrectly combine different publishers with similar names.
Outlier filtering may also remove valid records unintentionally.

## Did you make any assumptions in cleaning/transforming the data?
The notebook assumes years outside the range 1900–2026 are invalid.
It also assumes the first author or publisher in each list is the primary value.

## How was your data sourced/verified for credibility?
The data was sourced directly from the OpenLibrary public API, which is widely
used for bibliographic metadata.

## Was your data acquired in an ethical way?
Yes. The data was collected through publicly accessible API endpoints.

## How would you mitigate any ethical implications identified?
Manual review and additional validation checks could be added to reduce
incorrect fuzzy matching and filtering decisions.
